In [1]:
!pip install scikit-learn
!pip install pandas SQLAlchemy psycopg2-binary



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# If needed (uncomment to install):
# !pip install pandas SQLAlchemy psycopg2-binary

import re
from pathlib import Path

import pandas as pd
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg2 import sql
from sqlalchemy import create_engine

# === Config ===
csv_path = "sample_table/ICI_trials.csv"   # adjust if your file is elsewhere
pg_user = "postgres"
pg_password = "5555"
pg_host = "localhost"
pg_port = 5432
target_db = "mayo_interactive_table_db"
admin_db = "postgres"  # connect here first to ensure target DB exists

# === 1) Ensure database exists ===
conn = psycopg2.connect(
    dbname=admin_db,
    user=pg_user,
    password=pg_password,
    host=pg_host,
    port=pg_port,
)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()
cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (target_db,))
exists = cur.fetchone() is not None
if not exists:
    cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(target_db)))
cur.close()
conn.close()

# === 2) Read CSV and normalize column & table names ===
df = pd.read_csv(csv_path)

raw_name = Path(csv_path).stem
table_name = re.sub(r"[^a-zA-Z0-9_]+", "_", raw_name).lower().strip("_")
if not table_name or table_name[0].isdigit():
    table_name = f"t_{table_name}"

orig_cols = list(df.columns)
safe_cols = [
    re.sub(r"[^a-zA-Z0-9_]+", "_", c).lower().strip("_") or f"col_{i}"
    for i, c in enumerate(orig_cols)
]
# ensure uniqueness after cleaning
seen = {}
final_cols = []
for c in safe_cols:
    if c not in seen:
        seen[c] = 0
        final_cols.append(c)
    else:
        seen[c] += 1
        final_cols.append(f"{c}_{seen[c]}")
df.columns = final_cols

# === 3) Load into Postgres ===
engine = create_engine(
    f"postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{target_db}"
)

# make sure schema exists (public is default but harmless to ensure)
with engine.begin() as con:
    con.exec_driver_sql("CREATE SCHEMA IF NOT EXISTS public")

df.to_sql(
    table_name,
    engine,
    schema="public",
    if_exists="replace",   # use "append" to add to an existing table
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Loaded {len(df):,} rows into {target_db}.public.{table_name}")
if orig_cols != final_cols:
    print("Column mapping (original -> normalized):")
    for o, n in zip(orig_cols, final_cols):
        if o != n:
            print(f"  {o!r} -> {n!r}")


FileNotFoundError: [Errno 2] No such file or directory: 'sample_table/ICI_trials.csv'

In [ ]:
# If needed (uncomment to install):
# !pip install pandas SQLAlchemy psycopg2-binary

import pandas as pd
from sqlalchemy import create_engine, text, inspect

# ====== Connection ======
PG_USER = "postgres"
PG_PASS = "5555"
PG_HOST = "localhost"
PG_PORT = 5432
PG_DB   = "mayo_interactive_table_db"
TABLE   = "public.ici_trials"   # update if your table name differs

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    future=True,
)

# Optional: sanity check that the table exists
insp = inspect(engine)
if not insp.has_table("ici_trials", schema="public"):
    print("⚠️  Warning: table public.ici_trials was not found. If your table name differs, update TABLE above.")

# ====== Handy queries (editable) ======
QUERIES = {
    # 1) PD1 in NSCLC (treatment + ICI names)
    "pd1_nsclc": """
        SELECT nct, pmid, authors, year, original_follow_up,
               treatment_regimen, name_of_ici, class_of_ici, cancer_type
        FROM public.ici_trials
        WHERE class_of_ici ~* '\\bpd-?1\\b'
          AND cancer_type ILIKE '%non%small%cell%lung%';
    """,

    # 2) Combination ICIs in melanoma (names + clinical setting)
    "combo_melanoma": """
        SELECT nct, pmid, authors, year, original_follow_up,
               name_of_ici, clinical_setting_in_relation_to_surgery,
               monotherapy_combination, cancer_type
        FROM public.ici_trials
        WHERE monotherapy_combination ILIKE 'combination'
          AND cancer_type ILIKE '%melanoma%';
    """,

    # 3) CTLA4 in GEJ/esophageal (primary endpoints + sample size)
    "ctla4_gej": """
        SELECT nct, pmid, authors, year, original_follow_up,
               primary_endpoint, total_sample_size, class_of_ici, cancer_type
        FROM public.ici_trials
        WHERE class_of_ici ~* 'ctla-?4'
          AND (cancer_type ILIKE '%gej%' OR cancer_type ILIKE '%esophageal%');
    """,

    # 4) Phase 3 monotherapy in urothelial/bladder with follow-up durations
    "p3_mono_urothelial": """
        SELECT nct, pmid, authors, year, original_follow_up,
               trial_phase, monotherapy_combination, cancer_type,
               follow_up_duration_for_primary_endpoints_overall_rx_control AS follow_up_durations
        FROM public.ici_trials
        WHERE trial_phase ILIKE 'phase 3'
          AND monotherapy_combination ILIKE 'monotherapy'
          AND (cancer_type ILIKE '%urothelial%' OR cancer_type ILIKE '%bladder%');
    """,

    # 5) Adjuvant (after surgery) in NSCLC
    "adjuvant_nsclc": """
        SELECT nct, pmid, authors, year, original_follow_up,
               clinical_setting_in_relation_to_surgery, cancer_type
        FROM public.ici_trials
        WHERE clinical_setting_in_relation_to_surgery ILIKE 'adjuvant'
          AND (cancer_type ILIKE '%nsclc%' OR cancer_type ILIKE '%non%small%cell%lung%');
    """,

    # 6) Treatment/comparator for PD1 and/or CTLA4 in RCC
    "pd1_ctla4_rcc": """
        SELECT nct, pmid, authors, year, original_follow_up,
               treatment_regimen, control_regimen, class_of_ici, cancer_type
        FROM public.ici_trials
        WHERE class_of_ici ~* '(pd-?1|ctla-?4)'
          AND (cancer_type ILIKE '%renal cell%' OR cancer_type ILIKE '%rcc%' OR cancer_type ILIKE '%kidney%');
    """,

    # 7) All info for PD-L1 inhibitors
    "pdl1_all": """
        SELECT *
        FROM public.ici_trials
        WHERE class_of_ici ~* 'pd-?l1';
    """,

    # 8) Trial names with total sample size >= 500 (handles '500+' text)
    "n_ge_500": """
        SELECT nct, pmid, authors, year, original_follow_up, study_name, total_sample_size
        FROM public.ici_trials
        WHERE COALESCE(NULLIF(regexp_replace(total_sample_size::text, '[^0-9]', '', 'g'), '')::int, 0) >= 500;
    """,

    # 9) Treatment names where primary endpoint is OS
    "os_primary": """
        SELECT nct, pmid, authors, year, original_follow_up, treatment_regimen, primary_endpoint
        FROM public.ici_trials
        WHERE primary_endpoint ILIKE 'os'
           OR primary_endpoint ~* '\\boverall\\s+survival\\b';
    """,

    # 10) Trial names after 2020
    "year_gt_2020": """
        SELECT nct, pmid, authors, year, original_follow_up, study_name
        FROM public.ici_trials
        WHERE COALESCE(NULLIF(year::text, '')::int, 0) > 2020;
    """,
}

# ====== Helper to run either a keyed query or raw SQL ======
def run_sql(key_or_sql: str, params: dict | None = None, limit: int | None = 50) -> pd.DataFrame:
    """
    Run a saved query by key (from QUERIES) or a raw SQL string.
    Optionally appends a LIMIT if not already present.
    Returns a pandas DataFrame.
    """
    sql_str = QUERIES.get(key_or_sql, key_or_sql).strip().rstrip(";")
    # If the user didn't include a LIMIT and asked for one, append it
    if limit is not None and " limit " not in sql_str.lower():
        sql_str += f" LIMIT {int(limit)}"
    with engine.connect() as conn:
        df = pd.read_sql_query(text(sql_str), conn, params=params)
    print(f"✅ {len(df)} rows")
    return df

# ====== Examples (uncomment to run) ======
# df1 = run_sql("pd1_nsclc")
# df2 = run_sql("p3_mono_urothelial", limit=200)
import pandas as pd

if isinstance(df3, pd.DataFrame):
    cols = df3.columns.tolist()
else:
    # list-of-dicts case
    cols = list(df3[0].keys()) if len(df3) > 0 else []

print(cols)


In [ ]:
"""
THIS WILL GENERATE SQLs
Pipeline 3 - Filter names + column names on first stage before filter selection based on names.
"""


import os, time, random, json, ast
import re, textwrap
import pandas as pd
from dotenv import load_dotenv

import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError

# ============================
# CONFIG
# ============================
EXCEL_PATH_IN  = "ground_truths/gtsqls-vs-runsql.xlsx"
EXCEL_PATH_OUT = "runs/query-chosen-filters-SQL_version3.xlsx"
SHEET_NAME     = 0
QUESTION_COLUMN = "question"
RUNS_PER_QUESTION = 3
DELAY_BETWEEN_CALLS_SEC = 0.0
MAX_RETRIES = 3
MODEL_NAME = "gemini-2.0-flash"

FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"
COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"
FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"


REQUIRED_COLS = ["NCT", "PMID", "Authors", "Year"]
MAX_ADDITIONAL_COLS = 6

# ============================
# SETUP
# ============================
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_KEY environment variable is not set")

genai.configure(api_key=GEMINI_KEY)

with open(FILTER_NAMES_DEF_PATH, "r", encoding="utf-8") as f:
    FILTER_NAMES_TEXT = f.read()
with open(COLUMN_DEFS_PATH, "r", encoding="utf-8") as f:
    COLUMN_DEFS_TEXT = f.read()
with open(FILTER_DEFS_FULL_PATH, "r", encoding="utf-8") as f:
    FILTER_DEFS_TEXT = f.read()

# ============================
# HELPERS
# ============================

def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```") and s.endswith("```"):
        s = s.strip("`")
        s = "\n".join(s.splitlines()[1:])
    return s.strip()


def extract_json(text: str):
    raw = strip_code_fences(text or "")
    try:
        return json.loads(raw)
    except Exception:
        pass
    try:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = raw[start : end + 1]
            try:
                return json.loads(candidate)
            except Exception:
                try:
                    obj = ast.literal_eval(candidate)
                    if isinstance(obj, (dict, list)):
                        return obj
                except Exception:
                    pass
    except Exception:
        pass
    raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")


def call_model_with_retries(prompt: str, seed: int | None = None) -> str:
    attempts = max(1, MAX_RETRIES)
    model = genai.GenerativeModel(MODEL_NAME)
    delay = 1.2
    for attempt in range(1, attempts + 1):
        try:
            resp = model.generate_content(
                prompt,
            )
            txt = getattr(resp, "text", None)
            return (txt if txt is not None else str(resp)).strip()
        except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
            if attempt == attempts:
                return f"[ERROR after {attempt} attempts] {e}"
            time.sleep(delay * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.25)))
        except Exception as e:
            if attempt >= min(3, attempts):
                return f"[ERROR non-retryable? attempt {attempt}] {e}"
            time.sleep(delay * attempt)

def build_stage1_prompt(question: str) -> str:
    schema = ['nct', 'pmid', 'authors', 'year', 'original_follow_up', 'study_name', 'trial_phase', 'number_of_arms', 'originial_publication_or_follow_up', 'cancer_type', 'treatment_regimen', 'name_of_ici', 'class_of_ici', 'monotherapy_combination', 'type_of_combination', 'control_regimen', 'type_of_control', 'total_sample_size', 'lines_of_treatment', 'clincal_setting_in_relation_to_surgery', 'is_pd_l1_positivity_inclusion_criteria', 'is_any_other_biomarker_used_for_inclusion', 'primary_endpoint', 'priamry_multiple_composite_or_co_primary_endpoints', 'secondary_endpoint', 'type_of_follow_up_given', 'follow_up_duration_for_primary_endpoint_s_in_months', 'overall', 'rx', 'control']  # or df.head().to_string() to show rows

    return (
        "You are a medical expert researching cancer.\n"
        "You have access to the following SQL table schema (column names and types):\n"
        f"{schema}\n\n"
        f"Write a valid PostgreSQL query to answer the following question:\n"
        f"Question: {question}\n\n"
        "Rules:\n"
        "- Always SELECT the columns NCT, PMID, Authors, and Year unless the question explicitly says otherwise.\n"
        "Available filter CATEGORY NAMES (choose names only; do NOT assign values):\n"
        f"{FILTER_NAMES_DEF_PATH}\n\n"
        "Available columns and definitions:\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n\n"
        "- Add at most 6 additional relevant columns.\n"
        "- Apply WHERE filters only if clearly implied by the question.\n"
        "- Here are the relevant table attributes you must consider:"
        """
        # Database connection
        PG_USER = "postgres"
        PG_PASS = "5555"
        PG_HOST = "localhost"
        PG_PORT = 5432
        PG_DB   = "mayo_interactive_table_db"
        TABLE   = "public.ici_trials"  # optional sanity check below uses this
        """
        "- Return only the SQL (no prose, no JSON, no code fences)."
    )
def build_stage2_prompt(question: str) -> str:
    df = pd.read_excel("sample_table.xlsx")
    schema = ['nct', 'pmid', 'authors', 'year', 'original_follow_up', 'study_name', 'trial_phase', 'number_of_arms', 'originial_publication_or_follow_up', 'cancer_type', 'treatment_regimen', 'name_of_ici', 'class_of_ici', 'monotherapy_combination', 'type_of_combination', 'control_regimen', 'type_of_control', 'total_sample_size', 'lines_of_treatment', 'clincal_setting_in_relation_to_surgery', 'is_pd_l1_positivity_inclusion_criteria', 'is_any_other_biomarker_used_for_inclusion', 'primary_endpoint', 'priamry_multiple_composite_or_co_primary_endpoints', 'secondary_endpoint', 'type_of_follow_up_given', 'follow_up_duration_for_primary_endpoint_s_in_months', 'overall', 'rx', 'control']

    return (
        "You are a medical expert refining an SQL query.\n"
        "Here is the SQL table schema:\n"
        f"{schema}\n\n"
        f"Question: {question}\n\n"
        "Available filter categories and definitions:\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        "Write the final PostgreSQL query with specific WHERE filter values included.\n"
        """
        # Database connection
        PG_USER = "postgres"
        PG_PASS = "5555"
        PG_HOST = "localhost"
        PG_PORT = 5432
        PG_DB   = "mayo_interactive_table_db"
        TABLE   = "public.ici_trials"  # optional sanity check below uses this
        """
        "Rules:\n"
        "- Only include WHERE clauses that are justified by the question.\n"
        "- Return only the SQL (no prose, no JSON, no code fences)."
    )

# ============================
# NORMALIZATION & CONVERSION
# ============================

_FENCE_RE = re.compile(r"```(?:\w+)?\s*([\s\S]*?)```|~~~(?:\w+)?\s*([\s\S]*?)~~~", re.IGNORECASE)
_SQL_START_RE = re.compile(r"(?is)\b(with|select|insert|update|delete|create|alter|drop)\b")

def _normalize_sql(s: str) -> str:
    s = textwrap.dedent(s).strip()
    # collapse >2 consecutive blank lines
    s = re.sub(r"\n{3,}", "\n\n", s)
    if not s.endswith(";"):
        s += ";"
    return s

def extract_sqls(text: str) -> list[str]:

    if not text:
        return []

    out, seen = [], set()
    for m in _FENCE_RE.finditer(text):
        block = (m.group(1) or m.group(2) or "").strip()
        if _SQL_START_RE.search(block):
            sql = _normalize_sql(block)
            if sql not in seen:
                seen.add(sql)
                out.append(sql)


    if not out and _SQL_START_RE.search(text):
        parts = re.split(r";\s*(?:\n|\Z)", text)
        for p in parts:
            p = p.strip()
            if _SQL_START_RE.search(p):
                sql = _normalize_sql(p)
                if sql not in seen:
                    seen.add(sql)
                    out.append(sql)

    out = [re.sub(r"(?m)^\s*#.*$", "", s).strip() for s in out if s.strip()]
    return [s for s in out if s]


# ============================
# CORE RUNNERS
# ============================

def run_once_for_question(question: str, seed: int | None = None) -> str:
    p1 = build_stage1_prompt(question)
    sql = call_model_with_retries(p1, seed=seed)
    return sql.strip()

def run_n_times_for_question(question: str, n_runs: int):
    outputs = []
    for i in range(n_runs):
        seed = random.randint(1, 10_000_000)
        sql = run_once_for_question(question, seed=seed)
        outputs.append(sql)
        time.sleep(DELAY_BETWEEN_CALLS_SEC)
    return outputs


# ============================
# MAIN
# ============================

def main():
    df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)

    # auto-detect question column if needed
    global QUESTION_COLUMN
    if QUESTION_COLUMN not in df.columns:
        candidates = [c for c in df.columns if str(c).strip().lower() in {"question", "query", "prompt"}]
        if candidates:
            QUESTION_COLUMN = candidates[0]
        else:
            raise ValueError(
                f"Couldn't find a question column named '{QUESTION_COLUMN}'. Available columns: {list(df.columns)}"
            )


        run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
        for c in run_cols:
            if c not in df.columns:
                df[c] = ""

        for idx, row in df.iterrows():
            q = str(row[QUESTION_COLUMN]).strip()
            if not q or q.lower() == "nan":
                continue
            print(f"Processing row {idx}: {q[:80]}{'...' if len(q)>80 else ''}")
            merged_runs = run_n_times_for_question(q, RUNS_PER_QUESTION)
            for i, raw in enumerate(merged_runs):
                sql_list = extract_sqls(raw)
                df.at[idx, run_cols[i]] = (sql_list[0] if sql_list else "")



    # Save
    df.to_excel(EXCEL_PATH_OUT, index=False)
    print(f"Saved results to: {EXCEL_PATH_OUT}")


if __name__ == "__main__":
    main()

In [ ]:

# === Average precision/recall per question (across runs) ===
# Requirements: pandas, sqlalchemy, psycopg2-binary
import os
import math
import pandas as pd
from sqlalchemy import create_engine, text

# ---- DB Config (from user) ----
PG_USER = "postgres"
PG_PASS = "5555"
PG_HOST = "localhost"
PG_PORT = 5432
PG_DB   = "mayo_interactive_table_db"
EXCEL_PATH = "runs/query-chosen-filters-SQL_version3.xlsx"  # change if needed
RESULTS_DIR = "results"
OUT_XLSX = os.path.join(RESULTS_DIR, "avg_by_question.xlsx")

# ---- Helpers ----
def _normalize_value(v):
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(v, str):
        return v.strip()
    return v

def _align_and_to_set(df_pred: pd.DataFrame, df_gt: pd.DataFrame):
    if df_pred is None or df_gt is None:
        return set(), set(), []
    pred_cols = set(df_pred.columns)
    gt_cols = set(df_gt.columns)
    common_cols = list(pred_cols & gt_cols)

    if not common_cols and list(df_pred.columns) == list(df_gt.columns):
        common_cols = list(df_gt.columns)

    if not common_cols:
        return set(), set(), []

    pred_rows = set(tuple(_normalize_value(x) for x in row)
                    for row in df_pred[common_cols].itertuples(index=False, name=None))
    gt_rows   = set(tuple(_normalize_value(x) for x in row)
                    for row in df_gt[common_cols].itertuples(index=False, name=None))
    return pred_rows, gt_rows, common_cols

def precision_recall(df_pred: pd.DataFrame, df_gt: pd.DataFrame):
    pred_rows, gt_rows, common_cols = _align_and_to_set(df_pred, df_gt)

    if len(gt_rows) == 0 and len(pred_rows) == 0:
        return 1.0, 1.0
    if len(pred_rows) == 0 and len(gt_rows) > 0:
        return 0.0, 0.0
    if len(gt_rows) == 0 and len(pred_rows) > 0:
        return 0.0, 1.0

    inter = pred_rows & gt_rows
    prec = len(inter) / len(pred_rows) if len(pred_rows) > 0 else 0.0
    rec  = len(inter) / len(gt_rows)   if len(gt_rows) > 0 else 0.0
    return prec, rec

def _safe_sql(sql_text: str) -> bool:
    if not isinstance(sql_text, str):
        return False
    lowered = sql_text.strip().lower()
    forbidden = ["drop ", "truncate ", "delete ", "update ", "insert ", "alter "]
    return not any(tok in lowered for tok in forbidden)

# ---- Connect DB ----
engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}")

# ---- Load input Excel ----
df_input = pd.read_excel(EXCEL_PATH)

question_col = "Query"
gt_col = "Ground Truth SQL"
run_cols = [c for c in df_input.columns if c.lower().startswith("run_")]

rows_out = []

for idx, row in df_input.iterrows():
    question = row.get(question_col, None)
    gt_sql   = row.get(gt_col, None)

    # Execute GT
    df_gt = None
    gt_ok = False
    if isinstance(gt_sql, str) and _safe_sql(gt_sql):
        try:
            df_gt = pd.read_sql_query(text(gt_sql), con=engine)
            gt_ok = True
        except Exception as e:
            gt_ok = False

    # If GT failed, we can't compute metrics; leave NaNs
    if not gt_ok:
        rows_out.append({
            "Query": question,
            "Ground Truth SQL": gt_sql,
            "avg_precision": None,
            "avg_recall": None,
        })
        continue

    # Gather per-run precision/recall for this question
    prs = []
    rcs = []
    for run_name in run_cols:
        run_sql = row.get(run_name, None)
        if not (isinstance(run_sql, str) and _safe_sql(run_sql)):
            continue
        try:
            df_pred = pd.read_sql_query(text(run_sql), con=engine)
        except Exception:
            continue
        p, r = precision_recall(df_pred, df_gt)
        prs.append(p)
        rcs.append(r)

    avg_p = sum(prs)/len(prs) if prs else None
    avg_r = sum(rcs)/len(rcs) if rcs else None

    rows_out.append({
        "Query": question,
        "Ground Truth SQL": gt_sql,
        "avg_precision": avg_p,
        "avg_recall": avg_r,
    })

df_out = pd.DataFrame(rows_out, columns=["Query","Ground Truth SQL","avg_precision","avg_recall"])

os.makedirs(RESULTS_DIR, exist_ok=True)
df_out.to_excel(OUT_XLSX, index=False)

print(f"Saved per-question averages to: {OUT_XLSX}")
